# PolyChain: Polymer Property Prediction

End-to-end training and evaluation of the PolyChain model.

**Architecture:** Hierarchical Periodic Transformer with:
- GIN-S backbone (multi-scale graph encoding)
- HAMF (Hierarchy-Aware Multi-Scale Fusion)
- PECGN (Periodic Equivariant Chain-Growth Network)
- CST (Chain Statistics Token)

**Run time:** ~30-60 min for full 5-fold CV on T4 GPU.

## 1. Setup & Dependencies

In [ ]:
#@title 1.1 Mount Google Drive (for checkpoint persistence)
#@markdown This ensures your checkpoints survive Colab disconnections.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_PATH = "/content/drive/MyDrive/PolyChain"
!mkdir -p "$DRIVE_PATH/checkpoints" "$DRIVE_PATH/predictions" "$DRIVE_PATH/reports"
print(f"Drive workspace: {DRIVE_PATH}")

In [ ]:
#@title 1.2 Clone Repository
#@markdown Change the URL if your repo is private.
!git clone https://github.com/NotShubham1112/Poly.git 2>/dev/null || true
%cd Poly/polymer_competition
import os
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title 1.3 Install Dependencies
#@markdown Installs all required packages. Takes ~3-5 minutes.
!pip install -q torch torchvision
!pip install -q torch-geometric
!pip install -q rdkit
!pip install -q xgboost lightgbm catboost
!pip install -q scikit-learn pandas numpy scipy pyyaml tqdm
!pip install -q matplotlib seaborn shap optuna joblib
!pip install -q transformers tokenizers

# Verify installations
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")

from rdkit import Chem
import rdkit
print(f"RDKit: {rdkit.__version__}")

import torch_geometric
print(f"PyG: {torch_geometric.__version__}")

In [ ]:
#@title 1.4 Verify All Imports
import os, sys, json, pickle, time
import numpy as np
import pandas as pd
import yaml

# Project imports
from features.graphs import smiles_to_graph, kmer_graph, periodic_graph, BOND_FEAT_DIM
from features.graph_utils import build_multiscale, collate_multiscale
from models.polychain import PolyChain
from models.polychain.cst import compute_cst_batch, CST_DIM
from models.gnn import get_gnn
from models.mlp import FingerprintMLP
from training.train_utils import rmse, mae, r2_score, spearman, save_checkpoint, load_checkpoint

print("All imports successful!")
print(f"BOND_FEAT_DIM: {BOND_FEAT_DIM}")
print(f"CST_DIM: {CST_DIM}")

## 2. Dataset Preparation

In [ ]:
#@title 2.1 Load or Download Dataset
#@markdown Upload your train.csv to data/ or use the built-in polymer Tg dataset.

DATA_DIR = "data"
train_path = os.path.join(DATA_DIR, "train.csv")

if not os.path.exists(train_path):
    print("train.csv not found. Generating polymer Tg dataset...")
    !python data/download_polymer_data.py --dataset polymer_tg --data_dir {DATA_DIR}

train_df = pd.read_csv(train_path)
print(f"Dataset: {len(train_df)} samples")
print(f"Columns: {list(train_df.columns)}")
if "property" in train_df.columns:
    y = train_df["property"]
    print(f"\nTarget statistics:")
    print(f"  Mean: {y.mean():.3f}")
    print(f"  Std:  {y.std():.3f}")
    print(f"  Min:  {y.min():.3f}")
    print(f"  Max:  {y.max():.3f}")
train_df.head()

In [ ]:
#@title 2.2 Visualize Target Distribution
from reports.visualizations import ReportGenerator
gen = ReportGenerator("reports/plots")

if "property" in train_df.columns:
    gen.plot_target_distribution(train_df["property"].values, target_name="property")

In [ ]:
#@title 2.3 Generate CV Splits & Features
!python -m data.generate_splits --config config.yaml
!python -m features.build_features --config config.yaml

## 3. PolyChain Training

In [ ]:
#@title 3.1 Smoke Test: 100 Polymers, 3 Epochs
#@markdown Quick test to verify everything works before full training.
!python -m training.train --model_type polychain --fold 0 --person colab --max_samples 100 --epochs 3

In [ ]:
#@title 3.2 Verify Checkpoint & Run Inference
from inference.predictor import PolymerPredictor

ckpt_path = "outputs/checkpoints/colab_polychain_fold0_best.pt"
if os.path.exists(ckpt_path):
    predictor = PolymerPredictor(ckpt_path)
    test_smiles = ["*CCO*", "*CC(*)c1ccccc1*", "*CCl*"]
    preds = predictor.predict(test_smiles)
    for smi, pred in zip(test_smiles, preds):
        print(f"  {smi} -> {pred:.4f}")
else:
    print(f"Checkpoint not found: {ckpt_path}")
    print("Check training output above for errors.")

In [ ]:
#@title 3.3 Copy Checkpoint to Drive
!cp -v outputs/checkpoints/colab_polychain_fold0_*.pt "$DRIVE_PATH/checkpoints/" 2>/dev/null || echo "No checkpoints to copy"
print(f"Drive checkpoints: {os.listdir(DRIVE_PATH + '/checkpoints/')}")

## 4. Full 5-Fold Cross-Validation

In [ ]:
#@title 4.1 Run All Models (5-Fold CV)
#@markdown This trains all model types across 5 folds. Takes ~30-60 min.
#@markdown You can interrupt and resume -- checkpoints are saved incrementally.

#@markdown Select models to train:
MODELS = "ridge,rf,xgb,gcn,gat,polychain" #@param {type:"string"}

!python -m training.run_all_folds --models "$MODELS" --person colab --config config.yaml

In [ ]:
#@title 4.2 View Results Summary
summary_path = "results/summary.csv"
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    print("=" * 60)
    print("  MODEL COMPARISON (5-Fold CV)")
    print("=" * 60)
    print(summary.to_string(index=False))
else:
    print("No summary found. Run 4.1 first.")

In [ ]:
#@title 4.3 Generate Model Comparison Plot
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    model_rmse = dict(zip(summary["model"], summary["rmse_mean"]))
    gen.plot_model_comparison(model_rmse, metric_name="rmse")

## 5. PolyChain Ablation Study

In [ ]:
#@title 5.1 Run Ablation (Backbone, +HAMF, +PECGN, Full)
!python -m training.run_ablation --fold 0 --epochs 50 --config config.yaml

In [ ]:
#@title 5.2 View Ablation Results
ablation_path = "results/ablation_results.csv"
if os.path.exists(ablation_path):
    ablation = pd.read_csv(ablation_path)
    print("=" * 60)
    print("  POLYCHAIN ABLATION STUDY")
    print("=" * 60)
    print(ablation.to_string(index=False))

    variant_rmse = dict(zip(ablation["variant"], ablation["rmse"]))
    gen.plot_ablation(variant_rmse)

## 6. Generate Reports

In [ ]:
#@title 6.1 Generate All Reports
!python reports/generate_reports.py --config config.yaml --skip-shap

In [ ]:
#@title 6.2 Display Generated Plots
plot_dir = "reports/plots"
if os.path.exists(plot_dir):
    plots = sorted([f for f in os.listdir(plot_dir) if f.endswith(".png")])
    print(f"Generated {len(plots)} plots:")
    for p in plots:
        print(f"  - {p}")
else:
    print("No plots generated yet.")

## 7. Save Everything to Drive

In [ ]:
#@title 7.1 Sync All Results to Google Drive
!cp -rv outputs/checkpoints/* "$DRIVE_PATH/checkpoints/" 2>/dev/null || true
!cp -rv predictions/* "$DRIVE_PATH/predictions/" 2>/dev/null || true
!cp -rv reports/* "$DRIVE_PATH/reports/" 2>/dev/null || true
!cp -rv results/* "$DRIVE_PATH/reports/" 2>/dev/null || true

print(f"\nAll results synced to: {DRIVE_PATH}")
print(f"\nDrive contents:")
!ls -la "$DRIVE_PATH/checkpoints/" 2>/dev/null | head -20

## 8. Resume After Disconnection

In [ ]:
#@title 8.1 Resume Training from Drive Checkpoints
#@markdown If Colab disconnected, run this cell to resume.

# Copy checkpoints back from Drive
!cp -v "$DRIVE_PATH/checkpoints/"*.pt outputs/checkpoints/ 2>/dev/null || echo "No checkpoints in Drive"

# Resume Polychain training
!python -m training.train --model_type polychain --fold 0 --person colab --config config.yaml --resume

## 9. Inference Demo

In [ ]:
#@title 9.1 Predict on New Polymers
import glob
from inference.predictor import PolymerPredictor

best_ckpts = glob.glob("outputs/checkpoints/*polychain*best.pt")
if best_ckpts:
    predictor = PolymerPredictor(best_ckpts[0])

    test_polymers = {
        "Polyethylene (PE)": "*CC*",
        "Polystyrene (PS)": "*CC(*)c1ccccc1*",
        "PVC": "*CCl*",
        "PMMA": "*CC(=O)OC*",
        "Nylon 6,6": "*CC(=O)NCCCCCC(=O)N*",
    }

    print(f"{'Polymer':<20} {'SMILES':<30} {'Predicted Tg':>12}")
    print("-" * 65)
    for name, smi in test_polymers.items():
        pred = predictor.predict([smi])
        print(f"{name:<20} {smi:<30} {pred[0]:>12.1f}")
else:
    print("No trained checkpoint found. Run training first.")